# 2. RAG Revision

In [1]:
from src import FaqHttpLoader, RAGBase, OpenRouterClient
from minsearch import Index 

loader = FaqHttpLoader()
documents = loader.load()

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=OpenRouterClient(),
    llm_model="openrouter/owl-alpha",
    instructions=instructions,
)

Testing it

In [2]:
assistant.rag('How do I run Docker on Windows?')

'To run Docker on Windows, you can follow these steps based on your Windows version:\n\n**For Windows 10 Pro / 11 Pro**:\n- Ensure that Hyper-V is enabled, as Docker can use it as a backend. You can follow the [Enable Hyper-V Option on Windows 10 / 11](https://www.c-sharpcorner.com/article/install-and-configured-docker-desktop-in-windows-10/) tutorial for detailed instructions.\n\n**For Windows 10 Home / 11 Home**:\n- Since the \'Home\' version doesn\'t support Hyper-V, you should use WSL2 (Windows Subsystem for Linux). You can refer to [install WSL on Windows 11](https://pureinfotech.com/install-wsl-windows-11/) for detailed instructions.\n\nIf you encounter the "WslRegisterDistribution failed with error: 0x800701bc" error, you should update the WSL2 Linux Kernel by following the guidelines at [GitHub: WSL Issue 5393](https://github.com/microsoft/WSL/issues/5393).\n\nAdditionally, if you face the error "the input device is not a TTY" when using Docker run for Windows, you can resolve 

In [3]:
assistant.rag('How do I run ducker on windows?')

''

# 4. Function Calling [Ollama]

In [4]:
import requests
llm_model = "granite4.1:8b"

prompt = 'I just discovered the course. Can I join it?'
url = "http://localhost:11434/api/chat"
payload = {
    "model": llm_model,
    "messages": [
        {"role": "user", "content": prompt},
    ],
    "stream": False,
    "keep_alive": 0, # 0 = unload immediately
}
response = requests.post(url, json=payload, timeout=120)
response.raise_for_status()
data = response.json()
print(data["message"]["content"])

To determine whether you can join the course, I would need more information about the specific course in question. Could you please provide additional details such as:

1. The name or subject of the course.
2. The platform or institution offering the course (e.g., Coursera, edX, university website).
3. Any prerequisites or eligibility criteria mentioned for enrollment.
4. Whether the course is currently open for new registrations.

With this information, I can give you a more accurate answer regarding your ability to join the course.


Defining the tool

In [14]:
from pprint import pprint

# Correct format for Ollama /api/chat
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

developer_prompt = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.
If you look up information, use FAQ search.
""".strip()
user_prompt = 'I just discovered the course. Can I join it?'

chat_messages = [
    {'role': 'developer', 'content': developer_prompt},
    {'role': 'user', 'content': user_prompt}
]

payload = {
    "model": llm_model,
    "messages": chat_messages,
    "stream": False,
    "keep_alive": 0, # 0 = unload immediately
    "tools": [search_tool],   # <-- add here
}
response = requests.post(url, json=payload, timeout=120)
response.raise_for_status()
data = response.json()
msg = data["message"]
pprint(msg)

{'content': '',
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': {'query': 'join course'},
                              'index': 0,
                              'name': 'search'},
                 'id': 'call_212nos4s'}]}


Executing the function and sending the result back

In [15]:
import json

if msg.get("tool_calls"):
    print("Tool calls:")
    pprint(msg["tool_calls"])

tc = msg["tool_calls"][0]
fn_name = tc["function"]["name"]
fn_args = tc["function"]["arguments"]

# Выполняем поиск
results = assistant.search(**fn_args)
result_json = json.dumps(results, indent=2)

print(f"results found: {len(results)}")
# print("\nresult_json:"+"\n"+result_json)

Tool calls:
[{'function': {'arguments': {'query': 'join course'},
               'index': 0,
               'name': 'search'},
  'id': 'call_212nos4s'}]
results found: 5


Add the model's output to the conversation history. Then we add the tool result:

In [16]:
chat_messages.append(msg)

chat_messages.append({
    "type": "function_call_output",
    'call_id': tc['id'],
    'output': result_json,
})
# The call_id links the tool result to the specific function call the model requested. 
# If the model makes multiple function calls in one turn, each one gets its own call_id.

In [17]:
for m in chat_messages:
    print(m,"\n----------\n")

{'role': 'developer', 'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\nIf you look up information, use FAQ search."} 
----------

{'role': 'user', 'content': 'I just discovered the course. Can I join it?'} 
----------

{'role': 'assistant', 'content': '', 'tool_calls': [{'id': 'call_212nos4s', 'function': {'index': 0, 'name': 'search', 'arguments': {'query': 'join course'}}}]} 
----------

{'type': 'function_call_output', 'call_id': 'call_212nos4s', 'output': '[\n  {\n    "question": "Course: Can I still join the course after the start date?",\n    "answer": "Yes, even if you don\'t register, you\'re still eligible to submit the homework.\\n\\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don\'t leave everything for the last minute.",\n    "section": "General Course-Related Questions",\n    "course": "data-engineering-zoomcamp"\n  },\n  {\n    "question": 

Asking the model again

In [18]:
payload["messages"] = chat_messages
response = requests.post(url, json=payload, timeout=120)
response.raise_for_status()
data = response.json()
msg = data["message"]
pprint(msg)

{'content': 'Based on the information available, yes—new learners are welcome '
            'to enroll in the course at any time. If you have a specific '
            'section or program in mind, feel free to let me know so I can '
            'provide detailed enrollment steps or answer any additional '
            'questions you might have!',
 'role': 'assistant'}


In [19]:
pprint(msg["content"])

('Based on the information available, yes—new learners are welcome to enroll '
 'in the course at any time. If you have a specific section or program in '
 'mind, feel free to let me know so I can provide detailed enrollment steps or '
 'answer any additional questions you might have!')


# 5. The Agentic Loop [OpenRouter]

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",                          # на верхнем уровне, не внутри "function"
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {                            # тоже на верхнем уровне
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}
developer_prompt = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.
If you look up information, use FAQ search.
""".strip()
user_prompt = 'I just discovered the course. Can I join it?'

chat_messages = [
    {'role': 'developer', 'content': developer_prompt},
    {'role': 'user', 'content': user_prompt}
]

In [93]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

response = openai_client.responses.create(
    model='openrouter/owl-alpha', # openrouter/owl-alpha, z-ai/glm-5.1 , openai/gpt-oss-120b:free
    input=chat_messages,
    tools=[search_tool],
)

In [94]:
response.output_text

''

In [95]:
import json
from openai.types.responses import ResponseFunctionToolCall

call = next(
    item for item in response.output
    if isinstance(item, ResponseFunctionToolCall)
)
args = json.loads(call.arguments)
search = assistant.search
result = search(**args)
result_json = json.dumps(result, indent=2)

chat_messages.extend(response.output)
chat_messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

response2 = openai_client.responses.create(
    model='openrouter/owl-alpha',
    input=chat_messages,
    tools=[search_tool],
)
response2.output_text


"Yes, you can still join the course even after the start date! You're eligible to submit homework as long as the form is still open. However, keep in mind there are deadlines for the final projects, so don't leave everything to the last minute.\n\nTo get started:\n- Check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp) for the current cohort's details.\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.\n\nAlso, make sure you have the basic environment ready (Google Cloud account, Google Cloud SDK, Python 3, Terraform, Git) and review the prerequisites and syllabus to ensure you're comfortable with the topics."

A stronger developer prompt

In [107]:
developer_prompt = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches if needed. Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

chat_messages = [
    {'role': 'developer', 'content': developer_prompt},
    {'role': 'user', 'content': user_prompt}
]

In [108]:
chat_messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches if needed. Try to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

A generic function-call helper

In [110]:
def make_call(call):
    args = json.loads(call.arguments)
    f_name = call.name
    f = globals()[f_name]
    result = f(**args)
    result_json = json.dumps(result, indent=2)
    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

Processing one response

In [111]:
response = openai_client.responses.create(
    model='openrouter/owl-alpha',
    input=chat_messages,
    tools=[search_tool],
)

has_function_calls = False

for entry in response.output:
    chat_messages.append(entry)

    if entry.type == 'message':
        print(entry.content[0].text)

    if entry.type == 'function_call':
        print('function_call:', entry.name, entry.arguments)
        result = make_call(entry)
        chat_messages.append(result)
        has_function_calls = True

function_call: search {"query": "joining the course enrollment requirements"}


The full agent loop

In [113]:
while True:
    response = openai_client.responses.create(
        model='openrouter/owl-alpha',
        input=chat_messages,
        tools=[search_tool],
    )

    chat_messages.extend(response.output)
    has_function_calls = False

    for entry in response.output:
        if entry.type == 'message':
            print(entry.content[0].text)

        if entry.type == 'function_call':
            print('function_call:', entry.name, entry.arguments)
            result = make_call(entry)
            chat_messages.append(result)
            has_function_calls = True

    if not has_function_calls:
        break

Yes, you can join the course! Here are the key details:

**Registration:**
- You can still join even after the cohort starts, as long as the registration form remains open.
- Register via the link in the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp),
  but the specific start date and registration link for the current cohort are in the [README](https://github.com/DataTalksClub/data-engi
neering-zoomcamp/blob/main/README.md) of the course repository.
- Be aware of deadlines for submitting homework and the final project.

**Prerequisites:**
- Basic coding experience, familiarity with SQL, and Python experience (helpful, but not required).
- No prior data engineering experience is needed.

**Steps to follow:**
1. Visit the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp) for the current cohort's details.
2. Register before the cohort starts or as early as possible.
3. Join the [course Telegram channel](https://t.me/dezoomca

Wrapping it in a function

In [137]:
def agent_loop(question, model='arcee-ai/trinity-large-thinking:free'):
    chat_messages = [
        {'role': 'developer', 'content': developer_prompt},
        {'role': 'user', 'content': question}
    ]

    for i in range(10):
        print(f"\n--- iteration {i} ---")
        response = openai_client.responses.create(
            model=model,
            input=chat_messages,
            tools=[search_tool],
        )
        
        # print("output:", response.output)
        # print("output_text:", response.output_text)
        print("status:", response.status)

        # Исправление: model_dump() чтобы не было строк в истории
        chat_messages.extend(item.model_dump() for item in response.output)
        has_function_calls = False

        for entry in response.output:
            if entry.type == 'message':
                print(entry.content[0].text)
            if entry.type == 'function_call':
                print('function_call:', entry.name, entry.arguments)
                result = make_call(entry)
                chat_messages.append(result)
                has_function_calls = True

        if not has_function_calls:
            break

In [138]:
# baidu/cobuddy:free  arcee-ai/trinity-large-thinking:free  nvidia/nemotron-3-super-120b-a12b:free
# agent_loop('How do I run ducker on windows?')
agent_loop('How do I run ducker on windows?', model='arcee-ai/trinity-large-thinking:free')



--- iteration 0 ---
status: completed

I'll help you find information about running Docker on Windows. Let me search the FAQ for relevant entries.

function_call: search {"query":"Docker Windows"}

--- iteration 1 ---
status: completed


function_call: search {"query":"Docker Windows installation setup"}

--- iteration 2 ---
status: completed

Based on the FAQ search, I believe you meant **Docker** (not "ducker"). Docker is a platform for building, running, and managing containers. Here's how to get started with Docker on Windows:

## Installation

1. **Download Docker Desktop** from the official site: [Docker for Windows](https://docs.docker.com/desktop/install/windows-install/)
2. **System Requirements**:
   - Windows 10/11 64-bit
   - For **Windows 10/11 Pro**: Docker can use Hyper-V backend
   - For **Windows 10/11 Home**: Requires WSL2 (Windows Subsystem for Linux)

3. **Enable Required Features**:
   - WSL2 (for Home editions) or Hyper-V (for Pro editions)
   - Virtual machine p

In [139]:
agent_loop('I just discovered the course. Can I still join it?')


--- iteration 0 ---
status: completed


function_call: search {"query":"join late enrollment register course"}

--- iteration 1 ---
status: completed

Yes, you can still join the course even if you missed the start date! 

Based on the course policies, here's what you need to know:

- **You can register and participate** - It's not too late to join the course
- **Homework deadlines**: Some homework assignments may have already passed their deadlines, so you might not be able to submit all of them
- **Projects are still available**: You can still work on the course projects
- **Certificate eligibility**: You can still earn a certificate by submitting 2 out of 3 course projects and reviewing 3 peers by the deadline
- **Time management**: Be aware of upcoming deadlines and don't leave everything until the last minute

The course materials are typically available on-demand, so you can study at your own pace even if you join late.

Is there a specific course you're interested in, or would 

# 6. ToyAIKit framework

In [ ]:
#